# 自動化特徵工程 (Automated Feature Engineering)

本 Notebook 介紹現代自動化特徵工程工具和技術，幫助你快速生成高質量特徵。

## 涵蓋工具
1. **Featuretools** - 自動化深度特徵合成
2. **AutoFeat** - 基於線性模型的特徵生成
3. **tsfresh** - 時間序列特徵提取
4. **Category Encoders** - 高級類別編碼
5. **Feature-engine** - 特徵工程管道

## 為什麼需要自動化特徵工程？

### 手動特徵工程的挑戰
- ⏰ 耗時且重複性高
- 🧠 需要豐富的領域知識
- 🔍 容易遺漏有用的特徵組合
- 🔄 難以系統化和重用

### 自動化的優勢
- ✅ 快速生成大量候選特徵
- ✅ 系統化探索特徵空間
- ✅ 可重現和可擴展
- ✅ 釋放時間專注於模型優化

In [ ]:
# 安裝必要套件
!pip install featuretools autofeat tsfresh category_encoders feature-engine -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

import warnings
warnings.filterwarnings('ignore')

print("✅ 套件安裝完成")

## 1. Featuretools - 深度特徵合成 (Deep Feature Synthesis)

### 核心概念
- **實體 (Entity)**: 數據表
- **關係 (Relationship)**: 表之間的連接
- **原語 (Primitive)**: 特徵轉換函數
  - Transform Primitives: 單表轉換 (log, absolute, etc.)
  - Aggregation Primitives: 跨表聚合 (sum, mean, count, etc.)

### 使用場景
- 多表關聯數據
- 需要自動生成大量特徵
- 探索性特徵工程

In [ ]:
import featuretools as ft

# 創建示例數據：電商場景
np.random.seed(42)

# 客戶表
customers = pd.DataFrame({
    'customer_id': range(1, 101),
    'join_date': pd.date_range('2023-01-01', periods=100, freq='D'),
    'age': np.random.randint(18, 70, 100),
    'city': np.random.choice(['New York', 'Los Angeles', 'Chicago'], 100)
})

# 訂單表
orders = pd.DataFrame({
    'order_id': range(1, 501),
    'customer_id': np.random.choice(range(1, 101), 500),
    'order_date': pd.date_range('2023-01-01', periods=500, freq='H'),
    'amount': np.random.uniform(10, 500, 500)
})

# 訂單詳情表
order_items = pd.DataFrame({
    'item_id': range(1, 1501),
    'order_id': np.random.choice(range(1, 501), 1500),
    'product_id': np.random.choice(range(1, 51), 1500),
    'quantity': np.random.randint(1, 5, 1500),
    'price': np.random.uniform(5, 100, 1500)
})

print(f"客戶數: {len(customers)}")
print(f"訂單數: {len(orders)}")
print(f"訂單項目數: {len(order_items)}")

In [ ]:
# 創建 EntitySet（特徵工具的核心數據結構）
es = ft.EntitySet(id='ecommerce')

# 添加實體（表）
es = es.add_dataframe(
    dataframe_name='customers',
    dataframe=customers,
    index='customer_id',
    time_index='join_date'
)

es = es.add_dataframe(
    dataframe_name='orders',
    dataframe=orders,
    index='order_id',
    time_index='order_date'
)

es = es.add_dataframe(
    dataframe_name='order_items',
    dataframe=order_items,
    index='item_id'
)

# 定義關係
es = es.add_relationship('customers', 'customer_id', 'orders', 'customer_id')
es = es.add_relationship('orders', 'order_id', 'order_items', 'order_id')

print(es)

In [ ]:
# 自動生成特徵！
feature_matrix, feature_defs = ft.dfs(
    entityset=es,
    target_dataframe_name='customers',
    max_depth=2,  # 特徵合成深度
    verbose=True,
    n_jobs=-1  # 並行處理
)

print(f"\n生成了 {len(feature_defs)} 個特徵！")
print(f"\n特徵矩陣形狀: {feature_matrix.shape}")
print(f"\n前 10 個特徵：")
for i, feat in enumerate(feature_defs[:10]):
    print(f"{i+1}. {feat.get_name()}")

In [ ]:
# 查看生成的特徵
print("特徵預覽：")
feature_matrix.head()

### 自定義原語 (Custom Primitives)

In [ ]:
from featuretools.primitives import make_trans_primitive
from featuretools.variable_types import Numeric

# 創建自定義轉換原語
def is_weekend(column):
    """判斷是否為週末"""
    return column.dt.dayofweek >= 5

IsWeekend = make_trans_primitive(
    function=is_weekend,
    input_types=[ft.variable_types.Datetime],
    return_type=ft.variable_types.Boolean,
    description="是否為週末"
)

# 使用自定義原語生成特徵
feature_matrix_custom, feature_defs_custom = ft.dfs(
    entityset=es,
    target_dataframe_name='customers',
    trans_primitives=[IsWeekend, 'year', 'month', 'weekday'],
    max_depth=1
)

print("帶自定義原語的特徵：")
print([f.get_name() for f in feature_defs_custom if 'IS_WEEKEND' in f.get_name()])

## 2. AutoFeat - 基於線性模型的特徵生成

### 特點
- 🎯 自動生成多項式和交互特徵
- 🔍 基於 Lasso 自動特徵選擇
- 📊 適用於回歸和分類任務
- ⚡ 相對輕量快速

In [ ]:
from autofeat import AutoFeatRegressor, AutoFeatClassifier
from sklearn.datasets import make_classification

# 創建分類數據集
X, y = make_classification(
    n_samples=1000,
    n_features=10,
    n_informative=7,
    n_redundant=3,
    random_state=42
)

# 轉換為 DataFrame
feature_names = [f'feature_{i}' for i in range(X.shape[1])]
X_df = pd.DataFrame(X, columns=feature_names)

# 分割數據
X_train, X_test, y_train, y_test = train_test_split(
    X_df, y, test_size=0.2, random_state=42
)

print(f"訓練集形狀: {X_train.shape}")
print(f"測試集形狀: {X_test.shape}")

In [ ]:
# 使用 AutoFeat 進行自動特徵工程
afreg = AutoFeatClassifier(
    categorical_cols=[],
    feateng_steps=2,  # 特徵工程步驟數（1=線性, 2=二次）
    verbose=1
)

# 擬合和轉換
X_train_auto = afreg.fit_transform(X_train, y_train)
X_test_auto = afreg.transform(X_test)

print(f"\n原始特徵數: {X_train.shape[1]}")
print(f"生成後特徵數: {X_train_auto.shape[1]}")
print(f"\n選中的特徵: {afreg.good_cols_}")

In [ ]:
# 比較性能
from sklearn.linear_model import LogisticRegression

# 原始特徵
lr_original = LogisticRegression(max_iter=1000)
lr_original.fit(X_train, y_train)
y_pred_original = lr_original.predict(X_test)
acc_original = accuracy_score(y_test, y_pred_original)

# AutoFeat 特徵
lr_auto = LogisticRegression(max_iter=1000)
lr_auto.fit(X_train_auto, y_train)
y_pred_auto = lr_auto.predict(X_test_auto)
acc_auto = accuracy_score(y_test, y_pred_auto)

print(f"原始特徵準確率: {acc_original:.4f}")
print(f"AutoFeat 特徵準確率: {acc_auto:.4f}")
print(f"提升: {(acc_auto - acc_original) * 100:.2f}%")

## 3. Category Encoders - 高級類別編碼

### 支持的編碼方法
1. **Target Encoding** - 目標編碼
2. **Leave-One-Out Encoding** - 留一編碼
3. **CatBoost Encoding** - CatBoost 編碼
4. **James-Stein Encoding** - James-Stein 編碼
5. **WoE (Weight of Evidence)** - 證據權重編碼

In [ ]:
import category_encoders as ce

# 創建包含類別特徵的數據
df_cat = pd.DataFrame({
    'city': np.random.choice(['NYC', 'LA', 'Chicago', 'Houston'], 1000),
    'product': np.random.choice(['A', 'B', 'C', 'D', 'E'], 1000),
    'day_of_week': np.random.choice(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'], 1000),
    'price': np.random.uniform(10, 100, 1000),
    'quantity': np.random.randint(1, 10, 1000)
})

# 創建目標變量（銷售額）
df_cat['sales'] = (
    df_cat['price'] * df_cat['quantity'] +
    np.random.randn(1000) * 50
)

# 二分類目標
df_cat['high_sales'] = (df_cat['sales'] > df_cat['sales'].median()).astype(int)

print(df_cat.head())
print(f"\n數據形狀: {df_cat.shape}")

In [ ]:
# 準備數據
cat_features = ['city', 'product', 'day_of_week']
X_cat = df_cat[cat_features + ['price', 'quantity']]
y_cat = df_cat['high_sales']

X_cat_train, X_cat_test, y_cat_train, y_cat_test = train_test_split(
    X_cat, y_cat, test_size=0.2, random_state=42
)

In [ ]:
# 比較不同編碼方法
encoders = {
    'One-Hot': ce.OneHotEncoder(cols=cat_features),
    'Target': ce.TargetEncoder(cols=cat_features),
    'Leave-One-Out': ce.LeaveOneOutEncoder(cols=cat_features),
    'CatBoost': ce.CatBoostEncoder(cols=cat_features),
    'WoE': ce.WOEEncoder(cols=cat_features)
}

results = {}

for name, encoder in encoders.items():
    # 編碼
    X_train_enc = encoder.fit_transform(X_cat_train, y_cat_train)
    X_test_enc = encoder.transform(X_cat_test)
    
    # 訓練模型
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X_train_enc, y_cat_train)
    
    # 預測
    y_pred = rf.predict(X_test_enc)
    y_pred_proba = rf.predict_proba(X_test_enc)[:, 1]
    
    # 評估
    acc = accuracy_score(y_cat_test, y_pred)
    auc = roc_auc_score(y_cat_test, y_pred_proba)
    
    results[name] = {
        'accuracy': acc,
        'auc': auc,
        'n_features': X_train_enc.shape[1]
    }

# 顯示結果
results_df = pd.DataFrame(results).T
print("\n不同編碼方法的性能比較：")
print(results_df.sort_values('auc', ascending=False))

In [ ]:
# 可視化結果
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 準確率比較
results_df['accuracy'].sort_values().plot(kind='barh', ax=axes[0], color='skyblue')
axes[0].set_xlabel('Accuracy')
axes[0].set_title('編碼方法準確率比較')

# AUC 比較
results_df['auc'].sort_values().plot(kind='barh', ax=axes[1], color='lightcoral')
axes[1].set_xlabel('AUC')
axes[1].set_title('編碼方法 AUC 比較')

plt.tight_layout()
plt.show()

## 4. Feature-engine - 特徵工程管道

### 特點
- 🔧 與 sklearn pipeline 完全兼容
- 📦 包含豐富的轉換器
- 🎯 專注於特徵工程
- 🔄 易於集成到 ML 工作流

In [ ]:
from feature_engine.imputation import MeanMedianImputer, CategoricalImputer
from feature_engine.encoding import RareLabelEncoder, OrdinalEncoder
from feature_engine.transformation import LogTransformer, YeoJohnsonTransformer
from feature_engine.outliers import Winsorizer
from feature_engine.creation import MathFeatures
from sklearn.pipeline import Pipeline

# 創建帶有缺失值和異常值的數據
np.random.seed(42)
df_fe = pd.DataFrame({
    'age': np.random.randint(18, 80, 500),
    'income': np.random.lognormal(10, 1, 500),
    'credit_score': np.random.randint(300, 850, 500),
    'city': np.random.choice(['A', 'B', 'C', 'D', 'E', 'F'], 500),
    'education': np.random.choice(['HS', 'BS', 'MS', 'PhD'], 500)
})

# 引入缺失值
df_fe.loc[np.random.choice(df_fe.index, 50), 'income'] = np.nan
df_fe.loc[np.random.choice(df_fe.index, 30), 'city'] = np.nan

# 引入異常值
df_fe.loc[np.random.choice(df_fe.index, 10), 'income'] *= 10

# 創建目標變量
df_fe['target'] = (
    0.3 * df_fe['age'].fillna(df_fe['age'].mean()) +
    0.5 * np.log(df_fe['income'].fillna(df_fe['income'].median())) +
    0.2 * df_fe['credit_score'] +
    np.random.randn(500) * 100
) > 500

df_fe['target'] = df_fe['target'].astype(int)

print(f"數據形狀: {df_fe.shape}")
print(f"\n缺失值統計:")
print(df_fe.isnull().sum())

In [ ]:
# 構建特徵工程管道
from sklearn.model_selection import cross_val_score

# 定義特徵列
numeric_features = ['age', 'income', 'credit_score']
categorical_features = ['city', 'education']

# 創建管道
feature_pipeline = Pipeline([
    # 1. 處理缺失值
    ('numeric_imputer', MeanMedianImputer(
        imputation_method='median',
        variables=numeric_features
    )),
    ('categorical_imputer', CategoricalImputer(
        variables=categorical_features
    )),
    
    # 2. 處理異常值
    ('winsorizer', Winsorizer(
        capping_method='iqr',
        tail='both',
        fold=1.5,
        variables=['income']
    )),
    
    # 3. 處理稀有類別
    ('rare_label_encoder', RareLabelEncoder(
        tol=0.05,  # 出現頻率 < 5% 的視為稀有
        n_categories=3,
        variables=categorical_features
    )),
    
    # 4. 類別編碼
    ('ordinal_encoder', OrdinalEncoder(
        encoding_method='ordered',
        variables=categorical_features
    )),
    
    # 5. 數值轉換
    ('yeo_johnson', YeoJohnsonTransformer(
        variables=['income']
    )),
    
    # 6. 創建交互特徵
    ('math_features', MathFeatures(
        variables=[['age', 'credit_score']],
        func=['sum', 'prod', 'mean']
    ))
])

# 準備數據
X_fe = df_fe.drop('target', axis=1)
y_fe = df_fe['target']

# 應用管道
X_fe_transformed = feature_pipeline.fit_transform(X_fe, y_fe)

print(f"原始特徵數: {X_fe.shape[1]}")
print(f"轉換後特徵數: {X_fe_transformed.shape[1]}")
print(f"\n新增特徵: {list(set(X_fe_transformed.columns) - set(X_fe.columns))}")

In [ ]:
# 評估管道效果
from sklearn.ensemble import GradientBoostingClassifier

# 完整管道（特徵工程 + 模型）
full_pipeline = Pipeline([
    ('feature_engineering', feature_pipeline),
    ('model', GradientBoostingClassifier(n_estimators=100, random_state=42))
])

# 交叉驗證
scores = cross_val_score(
    full_pipeline, X_fe, y_fe,
    cv=5, scoring='roc_auc'
)

print(f"交叉驗證 AUC 分數: {scores}")
print(f"平均 AUC: {scores.mean():.4f} (+/- {scores.std() * 2:.4f})")

## 5. AI 輔助特徵工程

### 使用 AI 工具生成特徵工程代碼

In [ ]:
# AI 輔助特徵工程提示詞範例

prompt_examples = {
    "基礎特徵生成": """
    我有一個電商數據集，包含：
    - user_id: 用戶ID
    - purchase_date: 購買日期
    - amount: 購買金額
    - product_category: 產品類別
    
    請幫我生成以下特徵：
    1. 用戶總消費金額
    2. 用戶平均訂單金額
    3. 用戶購買頻率
    4. 距離上次購買的天數
    5. 最常購買的類別
    
    使用 pandas 和 feature_engine 實現。
    """,
    
    "時間序列特徵": """
    對於時間序列數據，請生成：
    1. 滾動窗口特徵（7天、30天平均）
    2. 滯後特徵（lag 1-7）
    3. 趨勢特徵（移動平均、指數平滑）
    4. 季節性特徵（月份、星期幾、節假日）
    5. 變化率特徵
    """,
    
    "類別特徵編碼": """
    我有高基數類別特徵（1000+ 唯一值）。
    請建議最佳編碼策略並實現：
    1. 處理稀有類別
    2. 選擇合適的編碼方法
    3. 避免過擬合
    4. 保持可解釋性
    
    使用 category_encoders 庫。
    """
}

print("💡 AI 輔助特徵工程提示詞範例：\n")
for name, prompt in prompt_examples.items():
    print(f"### {name}")
    print(prompt.strip())
    print("-" * 80)

## 6. 最佳實踐與工作流

### 特徵工程流程

```python
1. 數據理解
   ├─ 探索性數據分析 (EDA)
   ├─ 識別特徵類型
   └─ 發現數據質量問題

2. 基礎特徵工程
   ├─ 處理缺失值
   ├─ 處理異常值
   ├─ 類別編碼
   └─ 數值縮放

3. 自動化特徵生成
   ├─ Featuretools (多表關聯)
   ├─ AutoFeat (多項式特徵)
   ├─ tsfresh (時間序列)
   └─ 領域特定工具

4. 特徵選擇
   ├─ 過濾法 (Filter)
   ├─ 包裝法 (Wrapper)
   └─ 嵌入法 (Embedded)

5. 特徵驗證
   ├─ 交叉驗證
   ├─ 特徵重要性分析
   └─ 模型性能評估
```

### 工具選擇指南

| 場景 | 推薦工具 | 理由 |
|------|---------|------|
| 多表關聯數據 | Featuretools | 自動處理表關係和聚合 |
| 單表數據 | AutoFeat | 快速生成多項式特徵 |
| 時間序列 | tsfresh | 專門的時序特徵提取 |
| 類別編碼 | Category Encoders | 豐富的編碼方法 |
| 完整管道 | Feature-engine | sklearn 兼容性好 |
| 大規模數據 | Featuretools + Dask | 分散式處理 |

## 7. 總結

### 關鍵要點

1. **自動化不是萬能的**: 需要結合領域知識
2. **特徵數量 ≠ 模型性能**: 注重質量而非數量
3. **避免數據洩漏**: 在訓練/測試集分割後再做特徵工程
4. **可重現性**: 使用 Pipeline 確保一致性
5. **監控特徵**: 生產環境中追蹤特徵分佈變化

### 學習資源

- **Featuretools**: [官方文檔](https://www.featuretools.com/)
- **Category Encoders**: [GitHub](https://github.com/scikit-learn-contrib/category_encoders)
- **Feature-engine**: [官方文檔](https://feature-engine.readthedocs.io/)
- **書籍**: *Feature Engineering for Machine Learning* by Alice Zheng

### 下一步

1. 在實際項目中應用這些工具
2. 學習特徵選擇技術
3. 探索深度學習中的特徵學習
4. 掌握特徵存儲 (Feature Store) 概念